# ANGSI JR. V9 — JupyterLab Launch

Launch **from the same HPC environment that hosts the challenge data**. Run cells in order; do not use `%run angsi_jr_v9.py`.

In [ ]:
from pathlib import Path
import zipfile
root=Path('/data/epic20/APGCE/apgce_team_13')
archive=root/'ANGSI_JR_V9_ANALYSIS_FULL.zip'
assert archive.exists(), f'Upload ZIP to {root} first'
with zipfile.ZipFile(archive) as z:
    z.extractall(root)
app=root/'ANGSI_JR_V9_ANALYSIS'/'angsi_jr_v9.py'
print('App found:', app.exists(), app)


In [ ]:
%pip install -r /data/epic20/APGCE/apgce_team_13/ANGSI_JR_V9_ANALYSIS/requirements.txt


In [ ]:
import sys, os, subprocess, py_compile
py_compile.compile(str(app),doraise=True)
log=app.parent/'server_v9.log'
env=os.environ.copy()
env['ANGSI_DATA_ROOT']='/data/epic20/APGCE/input_data/challenge3'
with log.open('a') as output:
    process=subprocess.Popen([sys.executable,'-m','streamlit','run',str(app),
        '--server.address=127.0.0.1','--server.port=8513',
        '--server.headless=true','--server.fileWatcherType=none'],
        stdout=output,stderr=subprocess.STDOUT,cwd=app.parent,env=env,start_new_session=True)
print('Started PID',process.pid,'Log',log)


In [ ]:
import urllib.request
try:
    r=urllib.request.urlopen('http://127.0.0.1:8513/_stcore/health',timeout=6)
    print('Health',r.status,r.read().decode())
except Exception as exc:
    print('Health-check failed',type(exc).__name__,exc)
print('Last server logs:')
print(log.read_text(errors='replace')[-3500:] if log.exists() else 'No log yet')


Open `http://127.0.0.1:8513` **inside your HPC remote desktop**. JupyterHub proxy route `/user/apgce_team_13/proxy/8513/` works only when proxy is enabled. Tests: `PYTHONPATH=. python -m pytest -q tests/test_v9_analysis.py`.